Understanding the Attributes
FAMILY 1: Location & Identity (Where & When)
country, location_name

- **latitude, longitude**  
    Decimal degrees — the GPS coordinates of each city. Why these matter enormously:

    - Latitude is the strongest predictor of climate. The closer to the equator (lat ≈ 0), the warmer and wetter. Closer to the poles (lat ±90), the colder.
    - Longitude matters less for temperature but matters for things like "is this an oceanic or continental climate."
    - Together, they let us put the data on a world map — essential for spatial analysis.

timezone
IANA timezone name (e.g., Asia/Kabul). Useful for converting between local time and UTC.
last_updated_epoch vs. last_updated
This is subtle but important:

last_updated_epoch = Unix timestamp = UTC (universal time, same reference everywhere)
last_updated = same moment expressed in local time of that city


Why it matters: If we ask "what's happening globally at this moment?", we need _epoch. If we ask "what does temperature look like at 3pm local time across cities?", we need last_updated



FAMILY 2: Temperature (How Hot/Cold)
temperature_celsius / temperature_fahrenheit
Same value, two units. We'll keep Celsius and drop Fahrenheit (redundant).

What the numbers mean intuitively:
°CWhat it feels like< 0Freezing — water becomes ice0–10Cold — winter coat needed10–20Mild — light jacket20–25Comfortable — t-shirt weather25–32Warm to hot> 35Heat-wave territory> 40Dangerous heat


feels_like_celsius / feels_like_fahrenheit
This is the perceived temperature — what your body actually feels, accounting for:

Wind chill in cold weather (wind makes cold feel colder)
Heat index in warm weather (humidity makes hot feel hotter)




In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

### Loading Data

In [3]:
df = pd.read_csv("../data/GlobalWeatherRepository.csv")
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
df.head(3)

Shape: (138778, 41)
Memory: 122.69 MB


,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,wind_mph,wind_kph,wind_degree,wind_direction,pressure_mb,pressure_in,precip_mm,precip_in,humidity,cloud,feels_like_celsius,feels_like_fahrenheit,visibility_km,visibility_miles,uv_index,gust_mph,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,8.3,13.3,338,NNW,1012.0,29.89,0.0,0.0,24,30,25.3,77.5,10.0,6.0,7.0,9.5,15.3,277.0,103.0,1.1,0.2,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,6.9,11.2,320,NW,1012.0,29.88,0.1,0.0,94,75,19.0,66.2,10.0,6.0,5.0,11.4,18.4,193.6,97.3,0.9,0.1,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,9.4,15.1,280,W,1011.0,29.85,0.0,0.0,29,0,24.6,76.4,10.0,6.0,5.0,13.9,22.3,540.7,12.2,65.1,13.4,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55


### Drop Redundant Columns

In [4]:
# Keep metric units, drop imperial duplicates
redundant_cols = [
    'temperature_fahrenheit',
    'wind_mph',
    'pressure_in',
    'precip_in',
    'feels_like_fahrenheit',
    'visibility_miles',
    'gust_mph',
]
df = df.drop(columns=redundant_cols)
print(f"After dropping redundant cols: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

After dropping redundant cols: (138778, 34)
Memory: 114.92 MB


### Parse datetime + sort

In [5]:
df['last_updated'] = pd.to_datetime(df['last_updated'])
df = df.sort_values(['location_name', 'last_updated']).reset_index(drop=True)

print(f"Date range: {df['last_updated'].min()} → {df['last_updated'].max()}")
print(f"Total span: {(df['last_updated'].max() - df['last_updated'].min()).days} days")

Date range: 2024-05-16 01:45:00 → 2026-05-01 19:45:00
Total span: 715 days


### Drop duplicates

In [6]:
before = len(df)
df = df.drop_duplicates(subset=['location_name', 'last_updated'], keep='first')
print(f"Removed {before - len(df)} duplicate rows")

Removed 1 duplicate rows


### Cap physically-impossible values

In [7]:
# Define realistic physical bounds for each metric
physical_bounds = {
    'temperature_celsius':       (-90, 60),    # absolute records: -89°C, 56.7°C
    'feels_like_celsius':        (-100, 70),
    'wind_kph':                  (0, 410),     # max recorded ~408 km/h
    'gust_kph':                  (0, 510),     # gusts can exceed sustained wind
    'pressure_mb':               (870, 1085),  # realistic atmospheric range
    'humidity':                  (0, 100),
    'cloud':                     (0, 100),
    'precip_mm':                 (0, 500),     # extreme but possible
    'visibility_km':             (0, 50),
    'uv_index':                  (0, 20),
    'air_quality_PM2.5':         (0, 1000),    # cap at known disaster levels
    'air_quality_PM10':          (0, 2000),
    'air_quality_Carbon_Monoxide': (0, 50000),
    'air_quality_Ozone':         (0, 1000),
    'air_quality_Nitrogen_dioxide': (0, 500),
    'air_quality_Sulphur_dioxide': (0, 500),
}

print("OUTLIER CAPS (physical constraints):")
print("-" * 70)
for col, (lo, hi) in physical_bounds.items():
    if col in df.columns:
        out_low = (df[col] < lo).sum()
        out_high = (df[col] > hi).sum()
        if out_low + out_high > 0:
            print(f"{col:35s} | below {lo}: {out_low:5d} | above {hi}: {out_high:5d}")
        df[col] = df[col].clip(lower=lo, upper=hi)

OUTLIER CAPS (physical constraints):
----------------------------------------------------------------------
temperature_celsius                 | below -90:     0 | above 60:     1
feels_like_celsius                  | below -100:     0 | above 70:     1
wind_kph                            | below 0:     0 | above 410:     1
gust_kph                            | below 0:     0 | above 510:     1
pressure_mb                         | below 870:     0 | above 1085:     2
air_quality_PM2.5                   | below 0:     0 | above 1000:    10
air_quality_PM10                    | below 0:     2 | above 2000:   161
air_quality_Carbon_Monoxide         | below 0:     1 | above 50000:     0
air_quality_Sulphur_dioxide         | below 0:     1 | above 500:     1


### Cell 7: Statistical outlier flags

In [8]:
# Flag rows with IQR-based extreme values for later anomaly analysis
def flag_iqr_outliers(s, k=3.0):
    """Conservative IQR — k=3.0 catches only extreme outliers."""
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

iqr_cols = ['temperature_celsius', 'wind_kph', 'precip_mm',
            'air_quality_PM2.5', 'pressure_mb']

df['outlier_flag'] = False
for col in iqr_cols:
    if col in df.columns:
        flag = flag_iqr_outliers(df[col])
        df['outlier_flag'] |= flag
        print(f"{col:30s} | extreme outliers: {flag.sum():,}")

print(f"\nTotal rows flagged as containing extreme values: {df['outlier_flag'].sum():,} ({df['outlier_flag'].mean()*100:.1f}%)")

temperature_celsius            | extreme outliers: 73
wind_kph                       | extreme outliers: 126
precip_mm                      | extreme outliers: 24,618
air_quality_PM2.5              | extreme outliers: 5,178
pressure_mb                    | extreme outliers: 267

Total rows flagged as containing extreme values: 29,640 (21.4%)


### Add derived time features

In [9]:
df['year']        = df['last_updated'].dt.year
df['month']       = df['last_updated'].dt.month
df['day']         = df['last_updated'].dt.day
df['hour']        = df['last_updated'].dt.hour
df['day_of_year'] = df['last_updated'].dt.dayofyear
df['day_of_week'] = df['last_updated'].dt.dayofweek

def get_season(month, lat):
    """Northern vs Southern hemisphere seasons."""
    if lat >= 0:
        if month in [12, 1, 2]:  return 'Winter'
        if month in [3, 4, 5]:   return 'Spring'
        if month in [6, 7, 8]:   return 'Summer'
        return 'Autumn'
    else:
        if month in [12, 1, 2]:  return 'Summer'
        if month in [3, 4, 5]:   return 'Autumn'
        if month in [6, 7, 8]:   return 'Winter'
        return 'Spring'

df['season'] = df.apply(lambda r: get_season(r['month'], r['latitude']), axis=1)
print(df[['last_updated', 'year', 'month', 'season']].head())

         last_updated  year  month  season
0 2024-05-31 16:15:00  2024      5  Spring
1 2024-06-01 16:30:00  2024      6  Summer
2 2024-06-04 16:15:00  2024      6  Summer
3 2024-06-05 16:15:00  2024      6  Summer
4 2024-06-11 16:15:00  2024      6  Summer


### Add continent mapping

In [10]:
# Quick continent mapping using latitude/longitude
def lat_lon_to_continent(lat, lon):
    if lat > 35 and -25 < lon < 60:    return 'Europe'
    if lat > 10 and 25 < lon < 180:    return 'Asia'
    if lat < 35 and -20 < lon < 55:    return 'Africa'
    if -60 < lat < 15 and -90 < lon < -30: return 'South America'
    if 15 < lat < 75 and -170 < lon < -50: return 'North America'
    if lat < -10 and 110 < lon < 180:  return 'Oceania'
    if lat < -60:                      return 'Antarctica'
    return 'Other'

df['continent'] = df.apply(lambda r: lat_lon_to_continent(r['latitude'], r['longitude']), axis=1)
print(df['continent'].value_counts())

continent
Europe           40092
Africa           29918
Asia             26852
South America    15606
Other            14946
North America     8511
Oceania           2852
Name: count, dtype: int64


### Save the cleaned dataset

In [12]:
output_path = "../data/weather_cleaned.parquet"
df.to_parquet(output_path, index=False)

print(f"✅ Saved cleaned data to: {output_path}")
print(f"Shape: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
print(f"\nFinal columns ({len(df.columns)}):")
for c in df.columns:
    print(f"  • {c}")

ArrowKeyError: No type extension with name arrow.py_extension_type found

imports
shape
head
info
describe
datatype mismatch
null values
duplicate cols and rows
date range
understand data and business problem


In [12]:
df=pd.read_csv('../data/GlobalWeatherRepository.csv')
df.shape

(138778, 41)

In [10]:
pd.set_option('display.max_columns', None)
df.head(10)

,country,location_name,latitude,longitude,timezone,last_updated_epoch,last_updated,temperature_celsius,temperature_fahrenheit,condition_text,wind_mph,wind_kph,wind_degree,wind_direction,pressure_mb,pressure_in,precip_mm,precip_in,humidity,cloud,feels_like_celsius,feels_like_fahrenheit,visibility_km,visibility_miles,uv_index,gust_mph,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,sunrise,sunset,moonrise,moonset,moon_phase,moon_illumination
0,Afghanistan,Kabul,34.52,69.18,Asia/Kabul,1715849100,2024-05-16 13:15,26.6,79.8,Partly Cloudy,8.3,13.3,338,NNW,1012.0,29.89,0.00,0.00,24,30,25.3,77.5,10.0,6.0,7.0,9.5,15.3,277.0,103.0,1.1,0.2,8.4,26.6,1,1,04:50 AM,06:50 PM,12:12 PM,01:11 AM,Waxing Gibbous,55
1,Albania,Tirana,41.33,19.82,Europe/Tirane,1715849100,2024-05-16 10:45,19.0,66.2,Partly cloudy,6.9,11.2,320,NW,1012.0,29.88,0.10,0.00,94,75,19.0,66.2,10.0,6.0,5.0,11.4,18.4,193.6,97.3,0.9,0.1,1.1,2.0,1,1,05:21 AM,07:54 PM,12:58 PM,02:14 AM,Waxing Gibbous,55
2,Algeria,Algiers,36.76,3.05,Africa/Algiers,1715849100,2024-05-16 09:45,23.0,73.4,Sunny,9.4,15.1,280,W,1011.0,29.85,0.00,0.00,29,0,24.6,76.4,10.0,6.0,5.0,13.9,22.3,540.7,12.2,65.1,13.4,10.4,18.4,1,1,05:40 AM,07:50 PM,01:15 PM,02:14 AM,Waxing Gibbous,55
3,Andorra,Andorra La Vella,42.50,1.52,Europe/Andorra,1715849100,2024-05-16 10:45,6.3,43.3,Light drizzle,7.4,11.9,215,SW,1007.0,29.75,0.30,0.01,61,100,3.8,38.9,2.0,1.0,2.0,8.5,13.7,170.2,64.4,1.6,0.2,0.7,0.9,1,1,06:31 AM,09:11 PM,02:12 PM,03:31 AM,Waxing Gibbous,55
4,Angola,Luanda,-8.84,13.23,Africa/Luanda,1715849100,2024-05-16 09:45,26.0,78.8,Partly cloudy,8.1,13.0,150,SSE,1011.0,29.85,0.00,0.00,89,50,28.7,83.6,10.0,6.0,8.0,12.5,20.2,2964.0,19.0,72.7,31.5,183.4,262.3,5,10,06:12 AM,05:55 PM,01:17 PM,12:38 AM,Waxing Gibbous,55
5,Antigua and Barbuda,Saint John's,17.12,-61.85,America/Antigua,1715849100,2024-05-16 04:45,26.0,78.8,Partly cloudy,5.6,9.0,90,E,1013.0,29.91,0.02,0.00,84,25,28.2,82.8,10.0,6.0,1.0,15.7,25.3,220.3,29.0,0.2,0.2,1.2,4.5,1,1,05:36 AM,06:32 PM,01:05 PM,01:14 AM,Waxing Gibbous,55
6,Argentina,Buenos Aires,-34.59,-58.67,America/Argentina/Buenos_Aires,1715849100,2024-05-16 05:45,8.0,46.4,Clear,2.2,3.6,10,N,1014.0,29.94,0.00,0.00,93,0,7.1,44.9,10.0,6.0,1.0,6.5,10.5,270.4,7.7,8.3,1.3,4.0,5.3,1,1,07:43 AM,05:59 PM,02:36 PM,01:04 AM,Waxing Gibbous,55
7,Armenia,Yerevan,40.18,44.51,Asia/Yerevan,1715849100,2024-05-16 12:45,19.0,66.2,Partly cloudy,4.3,6.8,140,SE,1017.0,30.03,0.13,0.01,40,25,19.0,66.2,10.0,6.0,4.0,6.2,9.9,186.9,103.0,1.0,0.3,0.8,0.9,1,1,05:45 AM,08:12 PM,01:17 PM,02:31 AM,Waxing Gibbous,55
8,Australia,Canberra,-35.28,149.22,Australia/Sydney,1715849100,2024-05-16 18:45,9.0,48.2,Clear,2.5,4.0,100,E,1027.0,30.33,0.00,0.00,87,0,9.1,48.5,10.0,6.0,1.0,3.3,5.3,277.0,26.8,15.1,1.0,3.7,5.4,1,1,06:52 AM,05:07 PM,01:31 PM,No moonset,Waxing Gibbous,55
9,Austria,Vienna,48.20,16.37,Europe/Vienna,1715849100,2024-05-16 10:45,16.0,60.8,Partly cloudy,12.5,20.2,110,ESE,1013.0,29.91,0.00,0.00,63,75,16.0,60.8,10.0,6.0,5.0,19.8,31.9,220.3,68.7,5.1,4.1,3.7,4.4,1,1,05:14 AM,08:29 PM,01:00 PM,02:42 AM,Waxing Gibbous,55


In [15]:
df.describe()

,latitude,longitude,last_updated_epoch,temperature_celsius,temperature_fahrenheit,wind_mph,wind_kph,wind_degree,pressure_mb,pressure_in,precip_mm,precip_in,humidity,cloud,feels_like_celsius,feels_like_fahrenheit,visibility_km,visibility_miles,uv_index,gust_mph,gust_kph,air_quality_Carbon_Monoxide,air_quality_Ozone,air_quality_Nitrogen_dioxide,air_quality_Sulphur_dioxide,air_quality_PM2.5,air_quality_PM10,air_quality_us-epa-index,air_quality_gb-defra-index,moon_illumination
count,138778.000000,138778.000000,1.387780e+05,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000,138778.000000
mean,19.213299,21.949700,1.746734e+09,21.247973,70.248135,7.998991,12.876795,168.758802,1014.065176,29.944736,0.133593,0.005063,66.698288,39.905734,22.099157,71.773076,9.508937,5.617144,3.295366,11.317628,18.215840,456.050319,57.855360,14.919256,10.246664,24.039093,48.064901,1.693698,2.592522,49.736788
std,24.414296,65.786134,1.782547e+07,9.673907,17.412907,7.171831,11.538692,103.801547,10.274159,0.303347,0.559564,0.022117,23.842101,34.091593,11.578533,20.838992,2.692224,1.681689,3.544033,8.543596,13.749491,751.407805,30.614387,23.536538,35.255238,36.559110,148.127505,0.936784,2.432979,35.063794
min,-41.300000,-175.200000,1.715849e+09,-29.800000,-21.600000,2.200000,3.600000,1.000000,947.000000,27.960000,0.000000,0.000000,2.000000,0.000000,-36.700000,-34.000000,0.000000,0.000000,0.000000,2.200000,3.600000,-9999.000000,0.000000,0.000000,-9999.000000,0.168000,-1848.150000,1.000000,1.000000,0.000000
25%,4.050300,-6.836100,1.731316e+09,15.700000,60.300000,3.800000,6.100000,80.000000,1010.000000,29.830000,0.000000,0.000000,51.000000,0.000000,15.725000,60.300000,10.000000,6.000000,0.100000,6.400000,10.200000,199.800000,38.000000,1.700000,1.110000,7.050000,9.950000,1.000000,1.000000,15.000000
50%,17.250000,23.236100,1.746782e+09,23.800000,74.900000,6.700000,10.800000,161.000000,1013.000000,29.930000,0.000000,0.000000,72.000000,30.000000,25.100000,77.200000,10.000000,6.000000,1.800000,9.500000,15.300000,290.450000,55.000000,5.650000,2.405000,14.060000,19.795000,1.000000,2.000000,50.000000
75%,40.400000,49.882200,1.762155e+09,28.000000,82.400000,11.000000,17.600000,256.000000,1018.000000,30.060000,0.020000,0.000000,86.000000,75.000000,29.900000,85.800000,10.000000,6.000000,6.000000,15.000000,24.100000,456.950000,74.000000,17.205000,8.150000,27.600000,41.250000,2.000000,3.000000,85.000000
max,64.150000,179.220000,1.777618e+09,79.300000,174.700000,1841.200000,2963.200000,360.000000,3006.000000,88.770000,42.240000,1.660000,100.000000,100.000000,81.300000,178.300000,32.000000,19.000000,16.300000,1845.700000,2970.400000,38879.398000,480.700000,427.700000,521.330000,1614.100000,6037.290000,6.000000,10.000000,100.000000


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138778 entries, 0 to 138777
Data columns (total 41 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   country                       138778 non-null  object 
 1   location_name                 138778 non-null  object 
 2   latitude                      138778 non-null  float64
 3   longitude                     138778 non-null  float64
 4   timezone                      138778 non-null  object 
 5   last_updated_epoch            138778 non-null  int64  
 6   last_updated                  138778 non-null  object 
 7   temperature_celsius           138778 non-null  float64
 8   temperature_fahrenheit        138778 non-null  float64
 9   condition_text                138778 non-null  object 
 10  wind_mph                      138778 non-null  float64
 11  wind_kph                      138778 non-null  float64
 12  wind_degree                   138778 non-nul